In [1]:
# ==========================================================
# SRNN BRANCH PREDICTOR WITH FEDERATED LEARNING
# Based on: "A Dynamic Branch Predictor Based on Parallel Structure of SRNN"
# IEEE Access, Vol. 8, 2020 (pp. 86230-86237)
# Authors: Lei Zhang, Ning Wu, Fen Ge, Fang Zhou, Mahammad Rehan Yahya
# ==========================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from typing import List, Tuple, Optional
import math

# ==========================================================
# SETTINGS
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "6_clients"
FL_ROUNDS = 50
BATCH_SIZE = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================================
# SRNN CONFIGURATION (Based on Section IV Parameter Optimization)
# ==========================================================
class SRNNConfig:
    def __init__(self, 
                 ghr_length: int = 32,           # GHR register length (optimal from Fig 5)
                 pht_size: int = 1024,           # PHT table size (optimal from Fig 6)
                 slice_size: int = 6,            # Slice size for SRNN (optimal from Fig 8)
                 data_precision_bits: int = 8,   # Data precision (optimal from Fig 7)
                 threshold: int = 8):            # Update threshold θ (from Section III-B)
        """
        SRNN Branch Predictor Configuration
        
        Args:
            ghr_length: Global History Register length (bits)
            pht_size: Pattern History Table size (entries)
            slice_size: Slice size for SRNN parallelization
            data_precision_bits: Signed integer precision bits
            threshold: Update threshold θ (from training algorithm)
        """
        self.ghr_length = ghr_length
        self.pht_size = pht_size
        self.slice_size = slice_size
        self.data_precision_bits = data_precision_bits
        self.threshold = threshold
        
        # Data range for signed integers
        self.data_min = -(1 << (data_precision_bits - 1))
        self.data_max = (1 << (data_precision_bits - 1)) - 1
        
        # Number of layers in SRNN (computed based on slice size and history length)
        self.num_layers = self._compute_num_layers(ghr_length, slice_size)
        
        # Weight dimensions
        # W weights: one per history entry (size = ghr_length)
        # U weights: one per history entry minus one (for recurrent connections)
        self.w_size = ghr_length
        self.u_size = ghr_length - 1
        
    def _compute_num_layers(self, history_len: int, slice_size: int) -> int:
        """Compute number of SRNN layers needed"""
        if slice_size <= 0:
            return 1
        num_layers = 0
        remaining = history_len
        while remaining > 1:
            remaining = (remaining + slice_size - 1) // slice_size
            num_layers += 1
        return num_layers
    
    def get_total_storage_bits(self) -> int:
        """Calculate total storage in bits"""
        # PHT table: entries * (W weights + U weights)
        w_bits = self.w_size * self.data_precision_bits
        u_bits = self.u_size * self.data_precision_bits
        entry_bits = w_bits + u_bits
        
        # Total storage
        total = self.pht_size * entry_bits
        
        # GHR register
        total += self.ghr_length
        
        return total
    
    def get_total_storage_bytes(self) -> int:
        """Calculate total storage in bytes"""
        return self.get_total_storage_bits() // 8
    
    def get_total_storage_kb(self) -> float:
        """Calculate total storage in KB"""
        return self.get_total_storage_bits() / (8 * 1024)

# ==========================================================
# DATA LOADING
# ==========================================================
def load_csv(path):
    df = pd.read_csv(path)
    X = df.iloc[:, :-1].values.astype(np.int64)  # Branch PC (used as index)
    y = df.iloc[:, -1].values.astype(np.int64)   # Branch outcome (0/1)
    return X, y

def make_loader(X, y, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

# ==========================================================
# SRNN COMPUTATION ENGINE
# ==========================================================
class SRNNEngine:
    """
    SRNN (Sliced Recurrent Neural Network) Computation Engine
    Implements parallel RNN computation with slicing for reduced latency
    """
    
    def __init__(self, config: SRNNConfig):
        self.config = config
        
    def compute(self, 
                w_weights: np.ndarray, 
                u_weights: np.ndarray, 
                history: np.ndarray) -> int:
        """
        Compute SRNN prediction value y using parallel computation
        
        Args:
            w_weights: Input weights (size = history_length)
            u_weights: Recurrent weights (size = history_length - 1)
            history: Branch history (values: 1 for taken, -1 for not taken)
        
        Returns:
            y: Prediction value (positive = taken, negative = not taken)
        """
        # Step 1: First layer - Multiply inputs with W weights
        # S0[i] = f(x[i] * w[i]), where f(x) = x (linear activation)
        S0 = history * w_weights
        
        # Step 2: SRNN slicing and parallel computation
        current_layer = S0
        layer_size = len(current_layer)
        
        for layer_idx in range(self.config.num_layers):
            # Calculate number of groups in current layer
            slice_size = self.config.slice_size
            num_groups = (layer_size + slice_size - 1) // slice_size
            
            next_layer = []
            
            # Process each group in parallel
            for group_idx in range(num_groups):
                start_idx = group_idx * slice_size
                end_idx = min(start_idx + slice_size, layer_size)
                group_size = end_idx - start_idx
                
                if group_size == 1:
                    # Single element: pass through
                    next_layer.append(current_layer[start_idx])
                else:
                    # Multiple elements: compute RNN within group
                    # S_i = f(S_i-1 + u * S_prev)
                    group_result = current_layer[start_idx]  # First element
                    for i in range(start_idx + 1, end_idx):
                        u_idx = i - 1  # U weight index
                        if u_idx < len(u_weights):
                            group_result = current_layer[i] + u_weights[u_idx] * group_result
                    next_layer.append(group_result)
            
            current_layer = np.array(next_layer)
            layer_size = len(current_layer)
            
            # Stop if only one element remains
            if layer_size <= 1:
                break
        
        # Final output
        y = current_layer[0] if len(current_layer) > 0 else 0
        
        return int(y)
    
    def compute_parallel_pipeline(self, 
                                   w_weights: np.ndarray, 
                                   u_weights: np.ndarray, 
                                   history: np.ndarray) -> int:
        """
        Pipelined parallel computation (hardware-aware implementation)
        Each layer corresponds to one pipeline stage
        """
        # Stage 1: First layer multiplication
        S0 = history * w_weights
        
        # Pipeline registers for each layer
        current = S0
        size = len(current)
        
        # Stage 2+: SRNN layers (each stage is one pipeline cycle)
        for layer_idx in range(self.config.num_layers):
            slice_size = self.config.slice_size
            num_groups = (size + slice_size - 1) // slice_size
            
            next_stage = []
            
            for group_idx in range(num_groups):
                start = group_idx * slice_size
                end = min(start + slice_size, size)
                
                if end - start == 1:
                    next_stage.append(current[start])
                else:
                    # RNN computation within group
                    result = current[start]
                    for i in range(start + 1, end):
                        u_idx = i - 1
                        if u_idx < len(u_weights):
                            result = current[i] + u_weights[u_idx] * result
                    next_stage.append(result)
            
            current = np.array(next_stage)
            size = len(current)
            
            if size <= 1:
                break
        
        return int(current[0]) if len(current) > 0 else 0

# ==========================================================
# SRNN BRANCH PREDICTOR
# ==========================================================
class SRNNPredictor:
    """
    SRNN Branch Predictor (based on IEEE Access paper)
    
    Architecture:
    - Two-level predictor: GHR (Global History Register) + PHT (Pattern History Table)
    - PHT stores W and U weights for each branch PC
    - SRNN computes prediction using parallel sliced RNN structure
    """
    
    def __init__(self, config: SRNNConfig):
        self.config = config
        
        # Pattern History Table: stores weights for each branch PC
        # Each entry contains W weights (input) and U weights (recurrent)
        self.pht_w = {}  # Dict: PC -> W weights array
        self.pht_u = {}  # Dict: PC -> U weights array
        
        # Initialize weights to small random values
        self._initialize_weights()
        
        # Global History Register (stores branch outcomes: 1=taken, -1=not taken)
        self.ghr = np.ones(config.ghr_length) * -1  # Start with not taken
        self.ghr_idx = 0  # Current position in circular buffer
        
        # SRNN computation engine
        self.engine = SRNNEngine(config)
        
        # Statistics
        self.predictions = 0
        self.mispredictions = 0
        self.correct_predictions = 0
        
        # Update counter for threshold-based updates
        self.update_count = 0
        
    def _initialize_weights(self):
        """Initialize weights to small random values within data precision range"""
        for pc in range(self.config.pht_size):
            # Initialize W weights (input weights)
            w_weights = np.random.randint(-2, 3, size=self.config.w_size)
            w_weights = np.clip(w_weights, self.config.data_min, self.config.data_max)
            
            # Initialize U weights (recurrent weights)
            u_weights = np.random.randint(-2, 3, size=self.config.u_size)
            u_weights = np.clip(u_weights, self.config.data_min, self.config.data_max)
            
            self.pht_w[pc] = w_weights
            self.pht_u[pc] = u_weights
    
    def _get_pc_index(self, pc: int) -> int:
        """Get PHT index from PC (hash or modulo)"""
        return pc % self.config.pht_size
    
    def _get_history_array(self) -> np.ndarray:
        """Get current history as array (1=taken, -1=not taken)"""
        # Convert circular buffer to linear array (oldest to newest)
        history = np.roll(self.ghr, -self.ghr_idx)
        return history
    
    def _update_ghr(self, taken: bool):
        """Update Global History Register with new branch outcome"""
        value = 1 if taken else -1
        self.ghr[self.ghr_idx] = value
        self.ghr_idx = (self.ghr_idx + 1) % self.config.ghr_length
    
    def predict(self, pc: int) -> bool:
        """
        Predict branch direction using SRNN
        
        Returns: True = taken, False = not taken
        """
        self.predictions += 1
        
        # Get PHT index
        idx = self._get_pc_index(pc)
        
        # Get weights for this branch
        w_weights = self.pht_w.get(idx, np.zeros(self.config.w_size))
        u_weights = self.pht_u.get(idx, np.zeros(self.config.u_size))
        
        # Get current history
        history = self._get_history_array()
        
        # Compute prediction using SRNN
        y = self.engine.compute(w_weights, u_weights, history)
        
        # Prediction: taken if y > 0, not taken if y < 0
        # (if y == 0, default to not taken as per program bias)
        return y > 0
    
    def update(self, pc: int, actual: bool, predicted: bool = None):
        """
        Update predictor with actual branch outcome
        Implements the training algorithm from Section III-B
        
        Training rule:
        if |y| < θ or prediction != actual:
            for i in range(n):
                w_i = w_i + t * x_i
            for i in range(n-1):
                if x_i != t:
                    u_i = 1
                else:
                    u_i = u_i + 1
        """
        idx = self._get_pc_index(pc)
        
        # Get current weights
        w_weights = self.pht_w.get(idx, np.zeros(self.config.w_size)).copy()
        u_weights = self.pht_u.get(idx, np.zeros(self.config.u_size)).copy()
        
        # Get current history
        history = self._get_history_array()
        
        # Compute y value for threshold check
        y = self.engine.compute(w_weights, u_weights, history)
        
        # Convert actual to t (1 for taken, -1 for not taken)
        t = 1 if actual else -1
        
        # Get prediction if not provided
        if predicted is None:
            predicted = y > 0
        
        # Check if update is needed
        if abs(y) < self.config.threshold or predicted != actual:
            self.update_count += 1
            
            # Update W weights: w_i = w_i + t * x_i
            for i in range(len(w_weights)):
                w_weights[i] = w_weights[i] + t * history[i]
                # Clamp to data precision range
                w_weights[i] = np.clip(w_weights[i], self.config.data_min, self.config.data_max)
            
            # Update U weights
            for i in range(len(u_weights)):
                if history[i] != t:
                    u_weights[i] = 1
                else:
                    u_weights[i] = u_weights[i] + 1
                # Clamp to data precision range
                u_weights[i] = np.clip(u_weights[i], self.config.data_min, self.config.data_max)
            
            # Store updated weights
            self.pht_w[idx] = w_weights
            self.pht_u[idx] = u_weights
        
        # Update GHR
        self._update_ghr(actual)
        
        # Update statistics
        if predicted == actual:
            self.correct_predictions += 1
        else:
            self.mispredictions += 1
    
    def predict_and_update(self, pc: int, actual: bool) -> bool:
        """Make prediction and update with actual outcome"""
        pred = self.predict(pc)
        self.update(pc, actual, pred)
        return pred
    
    def get_accuracy(self) -> float:
        """Get prediction accuracy percentage"""
        if self.predictions == 0:
            return 0.0
        return (self.correct_predictions / self.predictions) * 100
    
    def get_mpki(self, instructions: int) -> float:
        """Calculate MPKI (Mispredictions Per Kilo Instructions)"""
        if instructions == 0:
            return 0.0
        return (self.mispredictions / instructions) * 1000
    
    def reset(self):
        """Reset predictor state"""
        self._initialize_weights()
        self.ghr = np.ones(self.config.ghr_length) * -1
        self.ghr_idx = 0
        self.predictions = 0
        self.mispredictions = 0
        self.correct_predictions = 0
        self.update_count = 0

# ==========================================================
# SRNN MODEL WRAPPER FOR FEDERATED LEARNING
# ==========================================================
class SRNNModel(nn.Module):
    """
    Wrapper to make SRNN compatible with PyTorch training loop
    """
    def __init__(self, config: SRNNConfig):
        super().__init__()
        self.config = config
        self.predictor = SRNNPredictor(config)
        
        # For compatibility with PyTorch
        self.parameters_list = nn.ParameterList([
            nn.Parameter(torch.tensor(0.0))
        ])
    
    def forward(self, x):
        """Forward pass for batch prediction"""
        batch_size = x.size(0)
        predictions = []
        
        for i in range(batch_size):
            pc = x[i, 0].item()
            pred = self.predictor.predict(pc)
            predictions.append(1.0 if pred else 0.0)
        
        return torch.tensor(predictions, device=x.device)
    
    def predict_and_update(self, pc: int, actual: bool) -> bool:
        """Predict and update with actual outcome"""
        return self.predictor.predict_and_update(pc, actual)
    
    def reset_state(self):
        """Reset predictor state"""
        self.predictor.reset()
    
    def get_accuracy(self) -> float:
        """Get prediction accuracy"""
        return self.predictor.get_accuracy()
    
    def get_mpki(self, instructions: int) -> float:
        """Get MPKI metric"""
        return self.predictor.get_mpki(instructions)

# ==========================================================
# FEDERATED LEARNING WITH SRNN
# ==========================================================
class FederatedSRNN:
    """Federated learning with SRNN predictors"""
    
    def __init__(self, config: SRNNConfig):
        self.config = config
        self.global_model = SRNNModel(config)
    
    def train_client(self, model: SRNNModel, data_loader: DataLoader):
        """Train client on their data stream"""
        model.reset_state()
        
        for Xb, yb in data_loader:
            for i in range(Xb.size(0)):
                pc = Xb[i, 0].item()
                actual = yb[i].item()
                model.predict_and_update(pc, actual)
        
        return model
    
    def aggregate_models(self, client_models: List[SRNNModel]):
        """Aggregate client models by averaging weights"""
        agg_model = SRNNModel(self.config)
        num_clients = len(client_models)
        
        # Average W weights for each PC
        for pc in range(self.config.pht_size):
            if pc not in agg_model.predictor.pht_w:
                continue
            
            # Sum W weights across clients
            sum_w = np.zeros(self.config.w_size)
            sum_u = np.zeros(self.config.u_size)
            
            for client_model in client_models:
                if pc in client_model.predictor.pht_w:
                    sum_w += client_model.predictor.pht_w[pc]
                    sum_u += client_model.predictor.pht_u[pc]
            
            # Average and round
            avg_w = np.round(sum_w / num_clients).astype(np.int64)
            avg_u = np.round(sum_u / num_clients).astype(np.int64)
            
            # Clamp to data range
            avg_w = np.clip(avg_w, self.config.data_min, self.config.data_max)
            avg_u = np.clip(avg_u, self.config.data_min, self.config.data_max)
            
            agg_model.predictor.pht_w[pc] = avg_w
            agg_model.predictor.pht_u[pc] = avg_u
        
        return agg_model

# ==========================================================
# EVALUATION FUNCTIONS
# ==========================================================
def evaluate_predictor(predictor: SRNNPredictor, data_loader: DataLoader) -> Tuple[float, float]:
    """Evaluate predictor accuracy and MPKI"""
    eval_predictor = SRNNPredictor(predictor.config)
    
    correct = 0
    total = 0
    mispredictions = 0
    
    for Xb, yb in data_loader:
        for i in range(Xb.size(0)):
            pc = Xb[i, 0].item()
            actual = yb[i].item()
            
            pred = eval_predictor.predict(pc)
            eval_predictor.update(pc, actual, pred)
            
            if pred == actual:
                correct += 1
            else:
                mispredictions += 1
            total += 1
    
    accuracy = (correct / total) * 100 if total > 0 else 0
    mpki = (mispredictions / total) * 1000 if total > 0 else 0
    
    return accuracy, mpki

# ==========================================================
# LOAD CLIENT DATA
# ==========================================================
train_files = sorted([f for f in os.listdir(DATA_DIR) if "train" in f])

client_train_loaders = []
client_test_loaders = []
client_test_labels = []

for f in train_files:
    X_train, y_train = load_csv(os.path.join(DATA_DIR, f))
    X_test, y_test = load_csv(os.path.join(DATA_DIR, f.replace("train", "test")))
    
    client_train_loaders.append(make_loader(X_train, y_train, shuffle=True))
    client_test_loaders.append(make_loader(X_test, y_test, shuffle=False))
    client_test_labels.append(y_test)

# Global test
X_global, y_global = load_csv(os.path.join(DATA_DIR, "test.csv"))
global_loader = make_loader(X_global, y_global, shuffle=False)

num_clients = len(client_train_loaders)

# ==========================================================
# INITIALIZE SRNN PREDICTOR (Optimal parameters from Section IV)
# ==========================================================
print(f"\n{'='*60}")
print(f"SRNN BRANCH PREDICTOR (IEEE Access 2020)")
print(f"Authors: Zhang, Wu, Ge, Zhou, Yahya")
print(f"{'='*60}")

# Use optimal parameters from the paper
config = SRNNConfig(
    ghr_length=32,      # Optimal from Fig 5
    pht_size=1024,      # Optimal from Fig 6
    slice_size=6,       # Optimal from Fig 8 (for GHR=32)
    data_precision_bits=8,  # Optimal from Fig 7
    threshold=8         # From Section III-B
)

print(f"\nSRNN Configuration (Optimal Parameters):")
print(f"  GHR length: {config.ghr_length} bits")
print(f"  PHT size: {config.pht_size} entries")
print(f"  Slice size: {config.slice_size}")
print(f"  Data precision: {config.data_precision_bits}-bit signed integer")
print(f"  Update threshold θ: {config.threshold}")
print(f"  Number of SRNN layers: {config.num_layers}")
print(f"  W weights per entry: {config.w_size}")
print(f"  U weights per entry: {config.u_size}")
print(f"  Total storage: {config.get_total_storage_kb():.2f} KB")
print(f"  Total storage: {config.get_total_storage_bytes()} bytes")

federated = FederatedSRNN(config)

# ==========================================================
# FEDERATED LEARNING LOOP
# ==========================================================
print(f"\n{'='*60}")
print(f"Starting Federated Learning with SRNN Branch Predictor")
print(f"{'='*60}")

global_accuracies = []
global_mpki_list = []

for rnd in range(1, FL_ROUNDS + 1):
    print(f"\n{'='*40}")
    print(f"ROUND {rnd}/{FL_ROUNDS}")
    print(f"{'='*40}")
    
    client_models = []
    client_accuracies = []
    client_mpki_list = []
    
    # Client training
    for c in range(num_clients):
        client_model = SRNNModel(config)
        trained_model = federated.train_client(client_model, client_train_loaders[c])
        
        accuracy, mpki = evaluate_predictor(trained_model.predictor, client_test_loaders[c])
        client_accuracies.append(accuracy)
        client_mpki_list.append(mpki)
        client_models.append(trained_model)
        
        print(f"Client {c:2d} | Test Acc: {accuracy:.2f}% | MPKI: {mpki:.2f}")
    
    # Average client stats
    avg_acc = np.mean(client_accuracies)
    avg_mpki = np.mean(client_mpki_list)
    print(f"\nAvg Client | Acc: {avg_acc:.2f}% | MPKI: {avg_mpki:.2f}")
    
    # Federated aggregation
    global_model = federated.aggregate_models(client_models)
    
    # Global test evaluation
    global_accuracy, global_mpki = evaluate_predictor(global_model.predictor, global_loader)
    global_accuracies.append(global_accuracy)
    global_mpki_list.append(global_mpki)
    
    print(f"\n>>> Global Model | Acc: {global_accuracy:.2f}% | MPKI: {global_mpki:.2f}")
    
    federated.global_model = global_model

# ==========================================================
# FINAL EVALUATION
# ==========================================================
print(f"\n{'='*60}")
print(f"FINAL EVALUATION")
print(f"{'='*60}")

# Per-client evaluation
print("\nPer-client evaluation with final global model:")
client_final_accs = []
client_final_mpki = []

for c in range(num_clients):
    accuracy, mpki = evaluate_predictor(global_model.predictor, client_test_loaders[c])
    client_final_accs.append(accuracy)
    client_final_mpki.append(mpki)
    print(f"Client {c:2d} | Acc: {accuracy:.2f}% | MPKI: {mpki:.2f}")

print(f"\n{'='*40}")
print(f"CLIENT AVERAGES")
print(f"{'='*40}")
print(f"Average Client Accuracy: {np.mean(client_final_accs):.2f}% ± {np.std(client_final_accs):.2f}")
print(f"Average Client MPKI: {np.mean(client_final_mpki):.2f} ± {np.std(client_final_mpki):.2f}")

# Global test evaluation
final_accuracy, final_mpki = evaluate_predictor(global_model.predictor, global_loader)
print(f"\n{'='*40}")
print(f"GLOBAL RESULTS")
print(f"{'='*40}")
print(f"Final Global Accuracy: {final_accuracy:.2f}%")
print(f"Final Global MPKI: {final_mpki:.2f}")

total_samples = len(y_global)
correct = int(final_accuracy * total_samples / 100)
print(f"\nDetailed Statistics:")
print(f"  Total samples: {total_samples}")
print(f"  Correct predictions: {correct}")
print(f"  Mispredictions: {total_samples - correct}")
print(f"  Update count (threshold-based): {global_model.predictor.update_count}")

# ==========================================================
# PERFORMANCE METRICS (Based on Paper Section V)
# ==========================================================
print(f"\n{'='*60}")
print(f"PERFORMANCE SUMMARY (vs Paper Results)")
print(f"{'='*60}")

print(f"\nSRNN Predictor Metrics:")
print(f"  Storage: {config.get_total_storage_kb():.2f} KB ({config.get_total_storage_bytes()} bytes)")
print(f"  GHR length: {config.ghr_length} bits")
print(f"  PHT entries: {config.pht_size}")
print(f"  Slice size: {config.slice_size}")
print(f"  SRNN layers: {config.num_layers}")
print(f"  Update threshold θ: {config.threshold}")

print(f"\nAccuracy Metrics:")
print(f"  Final Global Accuracy: {final_accuracy:.2f}%")
print(f"  Final Global MPKI: {final_mpki:.2f}")
print(f"  Average Client Accuracy: {np.mean(client_final_accs):.2f}%")

# Compare with paper claims
print(f"\n{'='*60}")
print(f"COMPARISON WITH PAPER CLAIMS")
print(f"{'='*60}")

print(f"\nPaper Claims (Section V):")
print(f"  - 2.34% higher accuracy than Perceptron in short learning period")
print(f"  - Higher accuracy than Bimodal and GShare under same hardware")
print(f"  - Good performance in short execution programs")

print(f"\nOur Results:")
print(f"  Accuracy: {final_accuracy:.2f}%")
print(f"  Training rounds: {FL_ROUNDS}")

# ==========================================================
# SAVE MODEL
# ==========================================================
model_path = "srrn_final.pth"
torch.save({
    'config': {
        'ghr_length': config.ghr_length,
        'pht_size': config.pht_size,
        'slice_size': config.slice_size,
        'data_precision_bits': config.data_precision_bits,
        'threshold': config.threshold
    },
    'pht_w': global_model.predictor.pht_w,
    'pht_u': global_model.predictor.pht_u,
    'ghr': global_model.predictor.ghr,
    'ghr_idx': global_model.predictor.ghr_idx,
    'final_accuracy': final_accuracy,
    'final_mpki': final_mpki
}, model_path)
print(f"\nModel saved to: {model_path}")

# ==========================================================
# WEIGHT DISTRIBUTION ANALYSIS
# ==========================================================
print(f"\n{'='*60}")
print(f"WEIGHT DISTRIBUTION ANALYSIS")
print(f"{'='*60}")

# Analyze W weights distribution
all_w_weights = []
all_u_weights = []

for pc in range(min(100, config.pht_size)):  # Sample first 100 entries
    if pc in global_model.predictor.pht_w:
        all_w_weights.extend(global_model.predictor.pht_w[pc])
        all_u_weights.extend(global_model.predictor.pht_u[pc])
 
if all_w_weights:
    print(f"\nW Weight Statistics (input weights):")
    print(f"  Mean: {np.mean(all_w_weights):.2f}")
    print(f"  Std: {np.std(all_w_weights):.2f}")
    print(f"  Min: {np.min(all_w_weights)}")
    print(f"  Max: {np.max(all_w_weights)}")

if all_u_weights:
    print(f"\nU Weight Statistics (recurrent weights):")
    print(f"  Mean: {np.mean(all_u_weights):.2f}")
    print(f"  Std: {np.std(all_u_weights):.2f}")
    print(f"  Min: {np.min(all_u_weights)}")
    print(f"  Max: {np.max(all_u_weights)}")

print(f"\n{'='*60}")
print(f"Training Complete!")
print(f"{'='*60}")


SRNN BRANCH PREDICTOR (IEEE Access 2020)
Authors: Zhang, Wu, Ge, Zhou, Yahya

SRNN Configuration (Optimal Parameters):
  GHR length: 32 bits
  PHT size: 1024 entries
  Slice size: 6
  Data precision: 8-bit signed integer
  Update threshold θ: 8
  Number of SRNN layers: 2
  W weights per entry: 32
  U weights per entry: 31
  Total storage: 63.00 KB
  Total storage: 64516 bytes

Starting Federated Learning with SRNN Branch Predictor

ROUND 1/50
Client  0 | Test Acc: 67.64% | MPKI: 323.61
Client  1 | Test Acc: 72.36% | MPKI: 276.39
Client  2 | Test Acc: 61.01% | MPKI: 389.91
Client  3 | Test Acc: 50.02% | MPKI: 499.82
Client  4 | Test Acc: 55.50% | MPKI: 444.98
Client  5 | Test Acc: 52.59% | MPKI: 474.10

Avg Client | Acc: 59.85% | MPKI: 401.47

>>> Global Model | Acc: 57.97% | MPKI: 420.27

ROUND 2/50
Client  0 | Test Acc: 67.79% | MPKI: 322.11
Client  1 | Test Acc: 72.42% | MPKI: 275.80
Client  2 | Test Acc: 61.18% | MPKI: 388.21
Client  3 | Test Acc: 50.52% | MPKI: 494.83
Client  4 | 